<a href="https://colab.research.google.com/github/jennifersolano-afk/AAI2025/blob/main/Coding_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # Allows the script to create plots on servers/Colab.

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


RANDOM_STATE = 42
OUTPUT_DIR = Path("assignment_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


def make_housing_data(n_records=250, random_state=RANDOM_STATE):
    """Create a realistic, reproducible housing dataset with 250 records."""
    rng = np.random.default_rng(random_state)
    locations = rng.choice(
        ["Downtown", "Suburb", "Rural"],
        size=n_records,
        p=[0.35, 0.45, 0.20],
    )
    square_footage = rng.integers(850, 3601, size=n_records)
    location_premium = {
        "Downtown": 175000,
        "Suburb": 85000,
        "Rural": -25000,
    }
    noise = rng.normal(0, 35000, size=n_records)
    price = (
        90000
        + square_footage * 185
        + pd.Series(locations).map(location_premium).to_numpy()
        + noise
    )

    return pd.DataFrame(
        {
            "square_footage": square_footage,
            "location": locations,
            "price": np.round(np.maximum(price, 100000), 0).astype(int),
        }
    )


def make_customer_data(n_records=300, random_state=RANDOM_STATE):
    """Create a realistic, reproducible customer dataset with 300 records."""
    rng = np.random.default_rng(random_state)
    contract = rng.choice(
        ["Month-to-month", "One year", "Two year"],
        size=n_records,
        p=[0.55, 0.25, 0.20],
    )
    internet_service = rng.choice(
        ["Fiber optic", "DSL", "No internet"],
        size=n_records,
        p=[0.45, 0.40, 0.15],
    )
    tenure = rng.integers(1, 73, size=n_records)
    monthly_charges = np.round(rng.normal(75, 25, size=n_records).clip(20, 150), 2)
    support_tickets = rng.poisson(1.5, size=n_records).clip(0, 8)
    senior_citizen = rng.choice([0, 1], size=n_records, p=[0.84, 0.16])

    # Churn probability intentionally reflects common business patterns:
    # short tenure, month-to-month contracts, high charges, and support issues.
    log_odds = (
        -2.1
        + 1.15 * (contract == "Month-to-month")
        - 0.65 * (contract == "Two year")
        + 0.55 * (internet_service == "Fiber optic")
        - 0.035 * tenure
        + 0.012 * (monthly_charges - 70)
        + 0.28 * support_tickets
        + 0.35 * senior_citizen
    )
    probability = 1 / (1 + np.exp(-log_odds))
    churn = rng.binomial(1, probability)

    return pd.DataFrame(
        {
            "tenure_months": tenure,
            "monthly_charges": monthly_charges,
            "support_tickets": support_tickets,
            "senior_citizen": senior_citizen,
            "contract": contract,
            "internet_service": internet_service,
            "churn": churn,
        }
    )


def part_1_house_price_prediction(housing):
    """Train linear regression and predict the requested Downtown home."""
    X = housing[["square_footage", "location"]]
    y = housing["price"]

    numeric_features = ["square_footage"]
    categorical_features = ["location"]
    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", "passthrough", numeric_features),
            (
                "categorical",
                OneHotEncoder(drop="first", handle_unknown="ignore"),
                categorical_features,
            ),
        ]
    )

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", LinearRegression()),
        ]
    )
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=RANDOM_STATE
    )
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    new_house = pd.DataFrame({"square_footage": [2000], "location": ["Downtown"]})
    predicted_price = model.predict(new_house)[0]
    feature_names = model.named_steps["preprocessor"].get_feature_names_out()
    coefficients = model.named_steps["regressor"].coef_

    print("PART 1: HOUSE PRICE PREDICTION")
    print(f"Records used: {len(housing)}")
    print(f"Predicted price for a 2,000 sq ft Downtown house: ${predicted_price:,.2f}")
    print(f"Test MAE: ${mean_absolute_error(y_test, predictions):,.2f}")
    print(f"Test R^2: {r2_score(y_test, predictions):.3f}")
    print("Coefficients (location effects are relative to Downtown):")
    for feature, coefficient in zip(feature_names, coefficients):
        print(f"  {feature}: ${coefficient:,.2f}")
    print(f"  Intercept: ${model.named_steps['regressor'].intercept_:,.2f}\n")


def part_2_customer_churn_prediction(customers):
    """Train logistic regression and classify a new customer's churn risk."""
    X = customers.drop(columns="churn")
    y = customers["churn"]
    numeric_features = [
        "tenure_months", "monthly_charges", "support_tickets", "senior_citizen"
    ]
    categorical_features = ["contract", "internet_service"]

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", StandardScaler(), numeric_features),
            ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ]
    )
    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            # Balancing helps the model detect the less frequent churn class.
            ("classifier", LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            )),
        ]
    )
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
    )
    model.fit(X_train, y_train)
    probabilities = model.predict_proba(X_test)[:, 1]
    predictions = (probabilities >= 0.50).astype(int)

    new_customer = pd.DataFrame(
        {
            "tenure_months": [5],
            "monthly_charges": [110.00],
            "support_tickets": [3],
            "senior_citizen": [0],
            "contract": ["Month-to-month"],
            "internet_service": ["Fiber optic"],
        }
    )
    churn_probability = model.predict_proba(new_customer)[0, 1]
    churn_class = int(churn_probability >= 0.50)

    print("PART 2: CUSTOMER CHURN PREDICTION")
    print(f"Records used: {len(customers)}")
    print(f"New customer churn probability: {churn_probability:.1%}")
    print(f"Predicted class using 0.50 threshold: {'Churn' if churn_class else 'Stay'}")
    print(f"Test accuracy: {accuracy_score(y_test, predictions):.3f}")
    print(f"Test ROC-AUC: {roc_auc_score(y_test, probabilities):.3f}")
    print(classification_report(y_test, predictions, zero_division=0))


def part_3_customer_segmentation(customers):
    """Use K-Means, select K with an elbow plot, and save cluster results."""
    segment_features = ["tenure_months", "monthly_charges", "support_tickets"]
    X = customers[segment_features]
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    k_values = range(2, 9)
    inertias = []
    for k in k_values:
        inertias.append(KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(X_scaled).inertia_)

    plt.figure(figsize=(7, 4))
    plt.plot(list(k_values), inertias, marker="o")
    plt.title("Elbow Method for Customer Segmentation")
    plt.xlabel("Number of clusters (K)")
    plt.ylabel("Within-cluster sum of squares")
    plt.xticks(list(k_values))
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "customer_segmentation_elbow.png", dpi=150)
    plt.close()

    # K=4 is selected because the elbow plot shows diminishing improvement
    # after four clusters while preserving useful business interpretability.
    selected_k = 4
    kmeans = KMeans(n_clusters=selected_k, random_state=RANDOM_STATE, n_init=10)
    customers = customers.copy()
    customers["cluster"] = kmeans.fit_predict(X_scaled)
    customers.to_csv(OUTPUT_DIR / "customer_segments.csv", index=False)

    summary = customers.groupby("cluster")[segment_features + ["churn"]].mean().round(2)
    print("PART 3: CUSTOMER SEGMENTATION")
    print(f"Selected K: {selected_k}; elbow plot saved to {OUTPUT_DIR / 'customer_segmentation_elbow.png'}")
    print(f"Cluster results saved to {OUTPUT_DIR / 'customer_segments.csv'}")
    print("\nAverage characteristics by cluster:")
    print(summary)
    print("\nMarketing suggestions:")
    for cluster, row in summary.iterrows():
        if row["churn"] >= 0.50:
            strategy = "Retention offer, service check-in, and contract upgrade incentive."
        elif row["monthly_charges"] >= summary["monthly_charges"].median():
            strategy = "Premium support, loyalty rewards, and targeted add-on offers."
        elif row["tenure_months"] >= summary["tenure_months"].median():
            strategy = "Loyalty appreciation campaign and referral benefits."
        else:
            strategy = "Onboarding education and low-cost introductory bundles."
        print(f"  Cluster {cluster}: {strategy}")


def main():
    housing = make_housing_data()
    customers = make_customer_data()
    housing.to_csv(OUTPUT_DIR / "housing_data.csv", index=False)
    customers.to_csv(OUTPUT_DIR / "customer_churn_data.csv", index=False)

    part_1_house_price_prediction(housing)
    part_2_customer_churn_prediction(customers)
    part_3_customer_segmentation(customers)


if __name__ == "__main__":
    main()


PART 1: HOUSE PRICE PREDICTION
Records used: 250
Predicted price for a 2,000 sq ft Downtown house: $633,888.65
Test MAE: $29,769.53
Test R^2: 0.948
Coefficients (location effects are relative to Downtown):
  numeric__square_footage: $183.21
  categorical__location_Rural: $-203,809.31
  categorical__location_Suburb: $-85,082.81
  Intercept: $267,465.93

PART 2: CUSTOMER CHURN PREDICTION
Records used: 300
New customer churn probability: 92.7%
Predicted class using 0.50 threshold: Churn
Test accuracy: 0.683
Test ROC-AUC: 0.768
              precision    recall  f1-score   support

           0       0.94      0.66      0.78        50
           1       0.32      0.80      0.46        10

    accuracy                           0.68        60
   macro avg       0.63      0.73      0.62        60
weighted avg       0.84      0.68      0.72        60

PART 3: CUSTOMER SEGMENTATION
Selected K: 4; elbow plot saved to assignment_outputs/customer_segmentation_elbow.png
Cluster results saved to as

In [13]:
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # Allows the script to create plots on servers/Colab.

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


RANDOM_STATE = 42
OUTPUT_DIR = Path("assignment_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


def make_housing_data(n_records=250, random_state=RANDOM_STATE):
    """Create a realistic, reproducible housing dataset with 250 records."""
    rng = np.random.default_rng(random_state)
    locations = rng.choice(
        ["Downtown", "Suburb", "Rural"],
        size=n_records,
        p=[0.35, 0.45, 0.20],
    )
    square_footage = rng.integers(850, 3601, size=n_records)
    location_premium = {
        "Downtown": 175000,
        "Suburb": 85000,
        "Rural": -25000,
    }
    noise = rng.normal(0, 35000, size=n_records)
    price = (
        90000
        + square_footage * 185
        + pd.Series(locations).map(location_premium).to_numpy()
        + noise
    )

    return pd.DataFrame(
        {
            "square_footage": square_footage,
            "location": locations,
            "price": np.round(np.maximum(price, 100000), 0).astype(int),
        }
    )


def make_customer_data(n_records=300, random_state=RANDOM_STATE):
    """Create a realistic, reproducible customer dataset with 300 records."""
    rng = np.random.default_rng(random_state)
    contract = rng.choice(
        ["Month-to-month", "One year", "Two year"],
        size=n_records,
        p=[0.55, 0.25, 0.20],
    )
    internet_service = rng.choice(
        ["Fiber optic", "DSL", "No internet"],
        size=n_records,
        p=[0.45, 0.40, 0.15],
    )
    tenure = rng.integers(1, 73, size=n_records)
    monthly_charges = np.round(rng.normal(75, 25, size=n_records).clip(20, 150), 2)
    support_tickets = rng.poisson(1.5, size=n_records).clip(0, 8)
    senior_citizen = rng.choice([0, 1], size=n_records, p=[0.84, 0.16])

    # Churn probability intentionally reflects common business patterns:
    # short tenure, month-to-month contracts, high charges, and support issues.
    log_odds = (
        -2.1
        + 1.15 * (contract == "Month-to-month")
        - 0.65 * (contract == "Two year")
        + 0.55 * (internet_service == "Fiber optic")
        - 0.035 * tenure
        + 0.012 * (monthly_charges - 70)
        + 0.28 * support_tickets
        + 0.35 * senior_citizen
    )
    probability = 1 / (1 + np.exp(-log_odds))
    churn = rng.binomial(1, probability)

    return pd.DataFrame(
        {
            "tenure_months": tenure,
            "monthly_charges": monthly_charges,
            "support_tickets": support_tickets,
            "senior_citizen": senior_citizen,
            "contract": contract,
            "internet_service": internet_service,
            "churn": churn,
        }
    )


def part_1_house_price_prediction(housing):
    """Train linear regression and predict the requested Downtown home."""
    X = housing[["square_footage", "location"]]
    y = housing["price"]

    numeric_features = ["square_footage"]
    categorical_features = ["location"]
    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", "passthrough", numeric_features),
            (
                "categorical",
                OneHotEncoder(drop="first", handle_unknown="ignore"),
                categorical_features,
            ),
        ]
    )

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", LinearRegression()),
        ]
    )
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=RANDOM_STATE
    )
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    new_house = pd.DataFrame({"square_footage": [2000], "location": ["Downtown"]})
    predicted_price = model.predict(new_house)[0]
    feature_names = model.named_steps["preprocessor"].get_feature_names_out()
    coefficients = model.named_steps["regressor"].coef_

    print("PART 1: HOUSE PRICE PREDICTION")
    print(f"Records used: {len(housing)}")
    print(f"Predicted price for a 2,000 sq ft Downtown house: ${predicted_price:,.2f}")
    print(f"Test MAE: ${mean_absolute_error(y_test, predictions):,.2f}")
    print(f"Test R^2: {r2_score(y_test, predictions):.3f}")
    print("Coefficients (location effects are relative to Downtown):")
    for feature, coefficient in zip(feature_names, coefficients):
        print(f"  {feature}: ${coefficient:,.2f}")
    print(f"  Intercept: ${model.named_steps['regressor'].intercept_:,.2f}\n")


def part_2_customer_churn_prediction(customers):
    """Train logistic regression and classify a new customer's churn risk."""
    X = customers.drop(columns="churn")
    y = customers["churn"]
    numeric_features = [
        "tenure_months", "monthly_charges", "support_tickets", "senior_citizen"
    ]
    categorical_features = ["contract", "internet_service"]

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", StandardScaler(), numeric_features),
            ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ]
    )
    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            # Balancing helps the model detect the less frequent churn class.
            ("classifier", LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            )),
        ]
    )
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
    )
    model.fit(X_train, y_train)
    probabilities = model.predict_proba(X_test)[:, 1]
    predictions = (probabilities >= 0.50).astype(int)

    new_customer = pd.DataFrame(
        {
            "tenure_months": [5],
            "monthly_charges": [110.00],
            "support_tickets": [3],
            "senior_citizen": [0],
            "contract": ["Month-to-month"],
            "internet_service": ["Fiber optic"],
        }
    )
    churn_probability = model.predict_proba(new_customer)[0, 1]
    churn_class = int(churn_probability >= 0.50)

    print("PART 2: CUSTOMER CHURN PREDICTION")
    print(f"Records used: {len(customers)}")
    print(f"New customer churn probability: {churn_probability:.1%}")
    print(f"Predicted class using 0.50 threshold: {'Churn' if churn_class else 'Stay'}")
    print(f"Test accuracy: {accuracy_score(y_test, predictions):.3f}")
    print(f"Test ROC-AUC: {roc_auc_score(y_test, probabilities):.3f}")
    print(classification_report(y_test, predictions, zero_division=0))


def part_3_customer_segmentation(customers):
    """Use K-Means, select K with an elbow plot, and save cluster results."""
    segment_features = ["tenure_months", "monthly_charges", "support_tickets"]
    X = customers[segment_features]
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    k_values = range(2, 9)
    inertias = []
    for k in k_values:
        inertias.append(KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(X_scaled).inertia_)

    plt.figure(figsize=(7, 4))
    plt.plot(list(k_values), inertias, marker="o")
    plt.title("Elbow Method for Customer Segmentation")
    plt.xlabel("Number of clusters (K)")
    plt.ylabel("Within-cluster sum of squares")
    plt.xticks(list(k_values))
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "customer_segmentation_elbow.png", dpi=150)
    plt.close()

    # K=4 is selected because the elbow plot shows diminishing improvement
    # after four clusters while preserving useful business interpretability.
    selected_k = 4
    kmeans = KMeans(n_clusters=selected_k, random_state=RANDOM_STATE, n_init=10)
    customers = customers.copy()
    customers["cluster"] = kmeans.fit_predict(X_scaled)
    customers.to_csv(OUTPUT_DIR / "customer_segments.csv", index=False)

    summary = customers.groupby("cluster")[segment_features + ["churn"]].mean().round(2)
    print("PART 3: CUSTOMER SEGMENTATION")
    print(f"Selected K: {selected_k}; elbow plot saved to {OUTPUT_DIR / 'customer_segmentation_elbow.png'}")
    print(f"Cluster results saved to {OUTPUT_DIR / 'customer_segments.csv'}")
    print("\nAverage characteristics by cluster:")
    print(summary)
    print("\nMarketing suggestions:")
    for cluster, row in summary.iterrows():
        if row["churn"] >= 0.50:
            strategy = "Retention offer, service check-in, and contract upgrade incentive."
        elif row["monthly_charges"] >= summary["monthly_charges"].median():
            strategy = "Premium support, loyalty rewards, and targeted add-on offers."
        elif row["tenure_months"] >= summary["tenure_months"].median():
            strategy = "Loyalty appreciation campaign and referral benefits."
        else:
            strategy = "Onboarding education and low-cost introductory bundles."
        print(f"  Cluster {cluster}: {strategy}")


def main():
    housing = make_housing_data()
    customers = make_customer_data()
    housing.to_csv(OUTPUT_DIR / "housing_data.csv", index=False)
    customers.to_csv(OUTPUT_DIR / "customer_churn_data.csv", index=False)

    part_1_house_price_prediction(housing)
    part_2_customer_churn_prediction(customers)
    part_3_customer_segmentation(customers)


if __name__ == "__main__":
    main()


PART 1: HOUSE PRICE PREDICTION
Records used: 250
Predicted price for a 2,000 sq ft Downtown house: $633,888.65
Test MAE: $29,769.53
Test R^2: 0.948
Coefficients (location effects are relative to Downtown):
  numeric__square_footage: $183.21
  categorical__location_Rural: $-203,809.31
  categorical__location_Suburb: $-85,082.81
  Intercept: $267,465.93

PART 2: CUSTOMER CHURN PREDICTION
Records used: 300
New customer churn probability: 92.7%
Predicted class using 0.50 threshold: Churn
Test accuracy: 0.683
Test ROC-AUC: 0.768
              precision    recall  f1-score   support

           0       0.94      0.66      0.78        50
           1       0.32      0.80      0.46        10

    accuracy                           0.68        60
   macro avg       0.63      0.73      0.62        60
weighted avg       0.84      0.68      0.72        60

PART 3: CUSTOMER SEGMENTATION
Selected K: 4; elbow plot saved to assignment_outputs/customer_segmentation_elbow.png
Cluster results saved to as

In [27]:
# Part 3: Customer Segmentation Using K-Means

import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans


# ---------------------------------------------------------
# DATA SOURCE:
# IBM Telco Customer Churn Dataset (Original source - not used directly)
# Local: /content/customer_churn_data.csv
# ---------------------------------------------------------


# Load the CSV file
df = pd.read_csv("/content/assignment_outputs/customer_churn_data.csv")


# ---------------------------------------------------------
# DATA CLEANING
# ---------------------------------------------------------

# 'monthly_charges' and other numeric features are already in correct format,
# and the generated data does not have 'TotalCharges' or missing values that need dropping.
# df["TotalCharges"] = pd.to_numeric(
#     df["TotalCharges"],
#     errors="coerce"
# )
# df = df.dropna()


# ---------------------------------------------------------
# SELECT FEATURES FOR CUSTOMER SEGMENTATION
# ---------------------------------------------------------

features = [
    "tenure_months",
    "monthly_charges",
    "support_tickets",
    "senior_citizen"
]

X = df[features].copy()

print("Number of customers:", len(X))

print("\nFeatures used for clustering:")
print(X.head())


# ---------------------------------------------------------
# SCALE THE FEATURES
# ---------------------------------------------------------

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# ---------------------------------------------------------
# ELBOW METHOD
# ---------------------------------------------------------

inertia = []
k_values = range(1, 9)

for k in k_values:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)


# ---------------------------------------------------------
# CREATE AND DISPLAY ELBOW PLOT
# ---------------------------------------------------------

plt.figure(figsize=(8, 5))

plt.plot(
    k_values,
    inertia,
    marker="o",
    linewidth=2
)

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for Customer Segmentation")
plt.xticks(k_values)
plt.grid(True)

plt.tight_layout()

# Save the chart
plt.savefig("elbow_plot.png", dpi=150)

# Make the chart appear
plt.show()


# ---------------------------------------------------------
# APPLY K-MEANS
# ---------------------------------------------------------

# Based on the elbow plot, K=3 is selected
k = 3

kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

# Create the Cluster column
df["Cluster"] = kmeans.fit_predict(X_scaled)


# ---------------------------------------------------------
# CREATE CUSTOMER SEGMENTATION CHART
# ---------------------------------------------------------

plt.figure(figsize=(8, 5))

scatter = plt.scatter(
    df["monthly_charges"],
    df["tenure_months"],
    c=df["Cluster"],
    cmap="viridis",
    s=50,
    alpha=0.8
)

plt.xlabel("Monthly Charges ($)")
plt.ylabel("Tenure (Months)")
plt.title("Customer Segmentation Using K-Means")
plt.colorbar(scatter, label="Cluster")
plt.grid(True)

plt.tight_layout()

# Save the chart
plt.savefig("customer_segmentation_chart.png", dpi=150)

# Make the chart appear
plt.show()


# ---------------------------------------------------------
# ANALYZE CLUSTERS
# ---------------------------------------------------------

cluster_summary = df.groupby("Cluster")[features].mean()

print("\nAverage Characteristics of Each Cluster:")
print(cluster_summary.round(2))


# ---------------------------------------------------------
# SHOW NUMBER OF CUSTOMERS IN EACH CLUSTER
# ---------------------------------------------------------

cluster_counts = df["Cluster"].value_counts().sort_index()

print("\nNumber of Customers in Each Cluster:")

for cluster, count in cluster_counts.items():
    print(f"Cluster {cluster}: {count} customers")


# ---------------------------------------------------------
# MARKETING STRATEGIES
# ---------------------------------------------------------

print("\nMarketing Strategies:")

for cluster in cluster_summary.index:

    monthly_charges = cluster_summary.loc[
        cluster, "monthly_charges"
    ]

    tenure = cluster_summary.loc[
        cluster, "tenure_months"
    ]

    print(f"\nCluster {cluster}:")
    print(f"Average monthly charges: ${monthly_charges:.2f}")
    print(f"Average tenure: {tenure:.1f} months")

    if monthly_charges > cluster_summary["monthly_charges"].median():

        print(
            "Strategy: Offer premium services, exclusive promotions, "
            "and loyalty rewards."
        )

    elif tenure < cluster_summary["tenure_months"].median():

        print(
            "Strategy: Offer onboarding support, discounts, "
            "and incentives for longer-term contracts."
        )

    else:

        print(
            "Strategy: Use personalized promotions and bundle offers "
            "to increase customer engagement."
        )


# ---------------------------------------------------------
# SAVE RESULTS TO CSV
# ---------------------------------------------------------

df.to_csv(
    "customer_segments.csv",
    index=False
)

print("\nCluster results saved to:")
print("customer_segments.csv")

Number of customers: 300

Features used for clustering:
   tenure_months  monthly_charges  support_tickets  senior_citizen
0             64            72.60                2               0
1             35           103.20                3               0
2             71            20.00                1               0
3             55            37.58                2               0
4             47            51.93                1               0

Average Characteristics of Each Cluster:
         tenure_months  monthly_charges  support_tickets  senior_citizen
Cluster                                                                 
0                56.69            75.05             1.39             0.0
1                19.81            76.13             1.62             0.0
2                34.31            73.16             1.42             1.0

Number of Customers in Each Cluster:
Cluster 0: 119 customers
Cluster 1: 129 customers
Cluster 2: 52 customers

Marketing Strategies:
